In [1]:
import os

data_dir = os.path.abspath('../data/FIVES')
gt_folder = os.path.join(data_dir, 'gt')
pred_folder = os.path.join(data_dir, 'pred')
img_folder = os.path.join(data_dir, 'img')

gt_path_list = os.listdir(gt_folder)
gt_path_list.sort()
print(len(gt_path_list))

800


In [4]:
from utils.reconstruction.reconstruction_method import PathReconstructionMethod

class PathReconstructionMethodInstance:
    def __init__(self, name: str, method: PathReconstructionMethod):
        self.name = name
        self.method = method

In [5]:
from utils.reconstruction.path_reconstruction import EuclideanPathReconstructionMethod, DahuDistancePathReconstructionMethod, ClassicMinEnergyPathReconstructionMethod, SquaredMinEnergyPathReconstructionMethod, MultiChannelSquaredMinEnergyPathReconstructionMethod
import torch

reconstruction_methods: list[PathReconstructionMethodInstance] = [
    PathReconstructionMethodInstance(
        "Euclidean", 
        EuclideanPathReconstructionMethod()
    ),

    PathReconstructionMethodInstance(
        "DahuDistance", 
        DahuDistancePathReconstructionMethod()
    ),
    
    PathReconstructionMethodInstance(
        "ClassicMinEnergy", 
        ClassicMinEnergyPathReconstructionMethod()
    ),

    PathReconstructionMethodInstance(
        "SquaredMinEnergy", 
        SquaredMinEnergyPathReconstructionMethod()
    )
]

In [6]:
import numpy as np

def compute_geodesic_length(path: np.ndarray) -> float:
    diffs = np.diff(path, axis=0)              # (N-1, 2)
    return np.linalg.norm(diffs, axis=1).sum()

def compute_geodesic_length_difference_metric(ref_path: np.ndarray,
                                     pred_path: np.ndarray
) -> float:
    ref_length = compute_geodesic_length(ref_path)
    pred_length = compute_geodesic_length(pred_path)
    return abs(ref_length - pred_length)

def compute_path_distances(ref_path: np.ndarray, 
                          pred_path: np.ndarray
) -> float:
    # pred_path: (P, 2)
    # ref_path:  (R, 2)

    diff = pred_path[:, None, :] - ref_path[None, :, :]   # (P, R, 2)
    dists = np.linalg.norm(diff, axis=2)                  # (P, R)

    min_dists = np.min(dists, axis=1)                      # (P,)

    average_min_distance = np.mean(min_dists)
    hausdorff_distance = np.max(min_dists)

    return average_min_distance, hausdorff_distance

def compute_metrics(ref_path, pred_path):
    if not isinstance(ref_path, np.ndarray):
        ref_path = np.array(ref_path)
    if not isinstance(pred_path, np.ndarray):
        pred_path = np.array(pred_path)

    ref_n_pixels = ref_path.shape[0] - 1
    euclidean_dist = np.linalg.norm(ref_path[0] - ref_path[-1])

    average_min_distance, hausdorff_distance = compute_path_distances(ref_path, pred_path)
    average_min_distance_inv, hausdorff_distance_inv = compute_path_distances(pred_path, ref_path)
    average_min_distance_sym = (average_min_distance + average_min_distance_inv) / 2.0
    hausdorff_distance_sym = max(hausdorff_distance, hausdorff_distance_inv)

    ref_length = compute_geodesic_length(ref_path)
    pred_length = compute_geodesic_length(pred_path)
    geodesic_length_diff_metric = abs(ref_length - pred_length)

    metrics = {
        "average_min_distance": average_min_distance_sym,
        "hausdorff_distance": hausdorff_distance_sym,
        "geodesic_length_diff_metric": geodesic_length_diff_metric
    }
    distances = {
        "euclidean": euclidean_dist,
        "n_pixels": ref_n_pixels,
        "geodesic": ref_length
    }

    return metrics, distances